# RB01：两连杆运动学与单关节动力学

配合 B2–B5；上传这一份 notebook 到 Colab，从干净 runtime 按顺序运行。数值部分只依赖 Python 标准库；可选图使用 IPython，缺少时跳过。没有外部数据、安装命令、模型权重或仓库路径依赖。

**当前 notebook 未执行，也未在 Colab 验证。** assert 是供运行时检查的教学条件，输出区保持空白。小模型不代表机器人仿真或硬件验证。

两连杆：q1 从基座 x 轴逆时针量，q2 相对第一杆，长度单位 m、角度用 rad。单摆部分另有角度零点。


In [ ]:
import math
import sys

print('Python:', sys.version.split()[0])
L1 = L2 = 1.0

def fk(q1, q2, l1=L1, l2=L2):
    x = l1 * math.cos(q1) + l2 * math.cos(q1 + q2)
    y = l1 * math.sin(q1) + l2 * math.sin(q1 + q2)
    return x, y

def joint_points(q1, q2, l1=L1, l2=L2):
    elbow = (l1 * math.cos(q1), l1 * math.sin(q1))
    return (0.0, 0.0), elbow, fk(q1, q2, l1, l2)


## 1. FK：先预测，再运行

两根 1 m 杆。分别预测 q=(0°,0°)、(0°,90°)、(90°,-90°) 的末端位置，以及 phi=q1+q2。最后两组是否只有位置相同？


In [ ]:
for degrees_pair in ((0, 0), (0, 90), (90, -90)):
    q1, q2 = map(math.radians, degrees_pair)
    print('q(deg)=', degrees_pair, 'position(m)=', fk(q1, q2),
          'orientation(deg)=', math.degrees(q1 + q2))

for actual, expected in zip(fk(0.0, math.pi / 2), (1.0, 1.0)):
    assert math.isclose(actual, expected, abs_tol=1e-10)


## 2. 可选示意图：相同目标，两种构型

左图 q=(0°,90°)，右图 q=(90°,-90°)。目标位置相同，但末端朝向不同。图中蓝色为第一杆，橙色为第二杆，绿色圈为末端。


In [ ]:
def arm_svg(poses):
    parts = ['<svg xmlns="http://www.w3.org/2000/svg" width="640" height="290" viewBox="0 0 640 290">',
             '<rect width="640" height="290" fill="white"/>']
    for index, (q1, q2, label) in enumerate(poses):
        origin_x, origin_y, scale = 160 + 320 * index, 230, 75
        def screen(point):
            return origin_x + scale * point[0], origin_y - scale * point[1]
        points = [screen(point) for point in joint_points(q1, q2)]
        parts.append(f'<text x="{320 * index + 20}" y="26" font-family="sans-serif" font-size="16" fill="#172b4d">{label}</text>')
        parts.append(f'<path d="M {origin_x-125} {origin_y} H {origin_x+135} M {origin_x} {origin_y+20} V {origin_y-170}" stroke="#b8c4d2" fill="none"/>')
        for start, end, color in ((points[0], points[1], '#2563eb'), (points[1], points[2], '#ea580c')):
            parts.append(f'<line x1="{start[0]}" y1="{start[1]}" x2="{end[0]}" y2="{end[1]}" stroke="{color}" stroke-width="7" stroke-linecap="round"/>')
        for x, y in points[:2]:
            parts.append(f'<circle cx="{x}" cy="{y}" r="6" fill="#172b4d"/>')
        x, y = points[2]
        parts.append(f'<circle cx="{x}" cy="{y}" r="8" fill="white" stroke="#15803d" stroke-width="3"/>')
        parts.append(f'<text x="{x-40}" y="{y-15}" font-family="sans-serif" font-size="14" fill="#15803d">tip (1, 1)</text>')
    parts.append('</svg>')
    return ''.join(parts)

try:
    from IPython.display import SVG, display
except ImportError:
    print('Optional SVG skipped: IPython is unavailable. Numeric sections still work.')
else:
    display(SVG(arm_svg([(0.0, math.pi/2, 'q = (0 deg, 90 deg)'),
                         (math.pi/2, -math.pi/2, 'q = (90 deg, -90 deg)')])))


## 3. IK：求两分支，再用 FK 验回去

这里只求位置，不加入朝向、限位或避障。不可达目标会报错；等长两杆在原点完全折叠的无限多解也单独报出，不冒充两组普通解。


In [ ]:
def ik(x, y, l1=L1, l2=L2, tolerance=1e-10):
    if l1 <= 0.0 or l2 <= 0.0:
        raise ValueError('Link lengths must be positive.')
    radius = math.hypot(x, y)
    if radius > l1 + l2 + tolerance or radius < abs(l1 - l2) - tolerance:
        raise ValueError('Unreachable target for this position task.')
    if radius < tolerance and abs(l1 - l2) < tolerance:
        raise ValueError('Degenerate folded case: infinitely many q1 values.')
    c2 = (x*x + y*y - l1*l1 - l2*l2) / (2.0*l1*l2)
    # 先做可达性判断，再处理边界附近的浮点舍入。
    if c2 < -1.0 - tolerance or c2 > 1.0 + tolerance:
        raise ValueError('Unreachable target: cosine outside range.')
    c2 = max(-1.0, min(1.0, c2))
    solutions = []
    for sign in (1.0, -1.0):
        s2 = sign * math.sqrt(max(0.0, 1.0 - c2*c2))
        q2 = math.atan2(s2, c2)
        q1 = math.atan2(y, x) - math.atan2(l2*s2, l1 + l2*c2)
        solutions.append((q1, q2))
    return solutions  # 边界上两分支可重合；这里没有枚举 2*pi 周期等价解。

for q1, q2 in ik(1.0, 1.0):
    reconstructed = fk(q1, q2)
    error = math.hypot(reconstructed[0]-1.0, reconstructed[1]-1.0)
    print('q(deg)=', (math.degrees(q1), math.degrees(q2)),
          'FK=', reconstructed, 'error(m)=', error)
    assert error < 1e-9

try:
    ik(3.0, 0.0)
except ValueError as error:
    print('Expected unreachable case:', error)


## 4. Jacobian：解析偏导与有限差分

先手算 q=(0,pi/2)、dq=(0.1,0.2) 时的末端速度。有限差分是验证局部导数的工具；它不是完整机械臂或 IK solver 的验证。


In [ ]:
def jacobian(q1, q2, l1=L1, l2=L2):
    s1, c1 = math.sin(q1), math.cos(q1)
    s12, c12 = math.sin(q1+q2), math.cos(q1+q2)
    return [[-l1*s1-l2*s12, -l2*s12],
            [l1*c1+l2*c12, l2*c12]]

def matvec(matrix, vector):
    return [sum(a*b for a, b in zip(row, vector)) for row in matrix]

def numerical_jacobian(q1, q2, epsilon=1e-6):
    q = [q1, q2]
    result = [[0.0, 0.0], [0.0, 0.0]]
    for column in range(2):
        plus, minus = q.copy(), q.copy()
        plus[column] += epsilon
        minus[column] -= epsilon
        p_plus, p_minus = fk(*plus), fk(*minus)
        for row in range(2):
            result[row][column] = (p_plus[row] - p_minus[row]) / (2.0*epsilon)
    return result

j = jacobian(0.0, math.pi/2)
print('J=', j, '| tip velocity(m/s)=', matvec(j, [0.1, 0.2]))
analytic = jacobian(0.3, 0.8)
numerical = numerical_jacobian(0.3, 0.8)
max_error = max(abs(analytic[r][c]-numerical[r][c]) for r in range(2) for c in range(2))
print('Derivative max error:', max_error)
assert max_error < 1e-6

j_straight = jacobian(0.0, 0.0)
determinant = j_straight[0][0]*j_straight[1][1] - j_straight[0][1]*j_straight[1][0]
print('Straight-arm J=', j_straight, '| determinant=', determinant)
assert math.isclose(determinant, 0.0, abs_tol=1e-12)

# 环境在末端施加向上的 10 N，映射为关节外力矩 J^T f。
f_external = [0.0, 10.0]
j_transpose = [list(column) for column in zip(*j)]
print('External joint torque(N m)=', matvec(j_transpose, f_external))


## 5. 动力学：改用一个单摆

q 从竖直向下方向逆时针量。I=0.5 kg m² 是相对于转轴的惯量，m=1 kg，质心距离=0.5 m，重力加速度=9.8 m/s²。没有外部接触，摩擦用 b*dq。

I*ddq + b*dq + m*g*lc*sin(q) = tau

先算水平静止、水平加速 2 rad/s²、水平零力矩三个情形。


In [ ]:
def inverse_dynamics(q, dq, ddq, inertia=0.5, mass=1.0, lc=0.5, gravity=9.8, damping=0.1):
    return inertia*ddq + damping*dq + mass*gravity*lc*math.sin(q)

def forward_dynamics(q, dq, torque, inertia=0.5, mass=1.0, lc=0.5, gravity=9.8, damping=0.1):
    if inertia <= 0.0:
        raise ValueError('Inertia must be positive.')
    return (torque - damping*dq - mass*gravity*lc*math.sin(q)) / inertia

q = math.pi / 2
hold = inverse_dynamics(q, 0.0, 0.0)
accelerate = inverse_dynamics(q, 0.0, 2.0)
fall_acceleration = forward_dynamics(q, 0.0, 0.0)
print('Hold torque(N m):', hold)
print('Torque for ddq=2(N m):', accelerate)
print('Zero-torque acceleration(rad/s^2):', fall_acceleration)
assert math.isclose(hold, 4.9)
assert math.isclose(accelerate, 5.9)
assert math.isclose(fall_acceleration, -9.8)
assert math.isclose(forward_dynamics(q, 0.0, accelerate), 2.0)


## 6. 只改一个数，留下自己的解释

1. FK：q=(pi/2,0) 在哪里？检查 sin/cos 接收的是 rad。
2. IK：目标改为 (1.5,0.5)，两组解是否都通过 FK 回代？若再指定朝向，两者是否仍可用？
3. Jacobian：q2 接近 0 时，看看 determinant 如何变化；区分瞬时不能向 x 运动与有限时间可以折叠。
4. 单摆：质量加倍而其他参数显式保持不变，静态保持力矩怎样变？真实改变质量也可能改变惯量，必须区分教学参数改动和物理改造。

运行日期 / runtime：

实际运行的 cell：

实际输出 / 修改的参数：

我的解释：

本 notebook 只计算教学模型，未实现数值 IK 迭代、完整控制器或机器人动力学仿真。

参考：[Modern Robotics FK](https://modernrobotics.northwestern.edu/chapters/chapter4/)、[IK](https://modernrobotics.northwestern.edu/chapters/chapter6/)、[Jacobian](https://modernrobotics.northwestern.edu/chapters/chapter5/)、[动力学](https://modernrobotics.northwestern.edu/chapters/chapter8/)。
